# Isolated iterations

One change at a time. Numbers come from `runs/iterations.jsonl`. The keep column is the decision in `runs/iterations.md`. This notebook does not refit.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from experiments.blocks import report

KEEP = {
    (1, "control_mi_half"): "no",
    (1, "corr_50"): "no",
    (1, "corr_100"): "no",
    (1, "corr_150"): "yes",
    (2, "corr_150"): "reference",
    (2, "corr_200"): "no",
    (2, "corr_250"): "no",
    (2, "corr_300"): "no",
    (3, "if_unscaled"): "no",
    (3, "no_row_drop"): "yes",
    (3, "if_after_zscore"): "no",
    (4, "single_xgb"): "reference",
    (4, "seed_mean"): "yes",
    (4, "xgb_ridge_mean"): "no",
    (5, "corr150"): "reference",
    (5, "corr150_nancount"): "no",
    (5, "corr150_miss10"): "no",
    (5, "corr150_both"): "no",
}
raw = [json.loads(line) for line in (ROOT / "runs/iterations.jsonl").read_text().splitlines() if line.strip()]
iters = pd.DataFrame([
    {
        "I": r["iteration"],
        "hyp": r["hyp"],
        "keep": KEEP[(r["iteration"], r["hyp"])],
        "n": r["n_train"],
        "p": r["n_features"],
        "ridge": r["ridge_cv_mean"],
        "xgb": r["xgb_cv_mean"],
    }
    for r in raw
])
report(iters, "Each try", bar="xgb")


I,hyp,keep,n,p,ridge,xgb,
1,control_mi_half,no,1192,412,0.066,0.487,
1,corr_50,no,1192,50,0.324,0.473,
1,corr_100,no,1192,100,0.319,0.498,
1,corr_150,yes,1192,150,0.299,0.519,
2,corr_150,reference,1192,150,0.299,0.519,
2,corr_200,no,1192,200,0.270,0.508,
2,corr_250,no,1192,250,0.256,0.508,
2,corr_300,no,1192,300,0.230,0.511,
3,if_unscaled,no,1192,150,0.299,0.519,
3,no_row_drop,yes,1212,150,0.278,0.542,


## What was kept

The accepted pipeline is no row drop, `|corr|` top-150, and a 3-seed XGBoost mean.


In [2]:
kept = iters[iters["keep"] == "yes"][["I", "hyp", "n", "p", "ridge", "xgb"]]
report(kept, "Kept")


I,hyp,n,p,ridge,xgb
1,corr_150,1192,150,0.299,0.519
3,no_row_drop,1212,150,0.278,0.542
4,seed_mean,1212,150,0.278,0.552
